# Data Cleaning và EDA — train_v2_drcat_02
Notebook gồm **2 phần**: làm sạch dữ liệu và khám phá dữ liệu văn bản.

Chạy các ô lần lượt từ trên xuống. Cần `pandas`, `numpy`, `matplotlib` và kernel Python (`ipykernel`). Nếu thiếu, chạy `%pip install pandas numpy matplotlib ipykernel` trong một ô mới.

Giữ nguyên file đầu vào. Kết quả được ghi vào `outputs/train_v2_drcat_02/` và được cập nhật khi chạy lại. Nhãn được gọi là **0/1** vì chưa có tài liệu xác nhận ý nghĩa; không suy ra nhãn từ tên nguồn.


## 1. Data Cleaning
### 1.1. Đọc dữ liệu và kiểm tra cấu trúc
Đọc chuỗi nguyên bản, không tự chuyển các chuỗi như “NA” thành giá trị thiếu. Dòng dữ liệu trống được giữ để kiểm tra. `row_id` là thứ tự bản ghi từ 1, không phải số dòng vật lý trong CSV.


In [ ]:
from pathlib import Path
import hashlib
import re
import unicodedata
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_PATH = Path('train_v2_drcat_02.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path.home() / 'OneDrive' / 'Máy tính' / 'train_v2_drcat_02.csv'
if not DATA_PATH.exists():
    raise FileNotFoundError('Sửa DATA_PATH để trỏ đến file CSV của bạn.')
OUT = DATA_PATH.parent / 'outputs' / 'train_v2_drcat_02'
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.figsize': (10, 5), 'axes.grid': False})
raw = pd.read_csv(DATA_PATH, dtype='string', keep_default_na=False, skip_blank_lines=False)
required = ['text', 'label', 'prompt_name', 'source', 'RDizzl3_seven']
assert set(required).issubset(raw.columns), f'Thiếu cột: {set(required) - set(raw.columns)}'
assert 'row_id' not in raw.columns, 'Cột row_id đã tồn tại; cần chọn tên định danh khác.'
df = raw.copy()
df.insert(0, 'row_id', np.arange(1, len(df) + 1))
print(f'Nguồn: {DATA_PATH.resolve()}\nKích thước: {raw.shape[0]:,} dòng × {raw.shape[1]} cột')
display(raw.head(3))
display(pd.DataFrame({'dtype': raw.dtypes.astype(str),
                    'blank': raw.apply(lambda s: s.str.strip().eq('').sum()),
                    'unique': raw.nunique()}))


### 1.2. Chuẩn hóa có kiểm soát
Giữ `text` gốc; tạo `text_clean` bằng Unicode NFC, chuẩn hóa CRLF/CR thành LF và bỏ khoảng trắng đầu/cuối. Giữ nguyên chữ hoa, dấu câu, lỗi chính tả và khoảng trắng bên trong vì chúng có thể mang thông tin về phong cách viết. Metadata được bỏ khoảng trắng đầu/cuối; giá trị gốc vẫn có trong CSV đầu vào.


In [ ]:
def normalize_text(value):
    return unicodedata.normalize('NFC', value).replace('\r\n', '\n').replace('\r', '\n').strip()

df['text_clean'] = df['text'].map(normalize_text)
for col in ['prompt_name', 'source']:
    df[col] = df[col].str.strip().replace('', pd.NA)
label_num = pd.to_numeric(df['label'].str.strip(), errors='coerce')
df['label'] = label_num.where(label_num.isin([0, 1])).astype('Int64')
bool_map = {'true': True, 'false': False}
df['RDizzl3_seven'] = df['RDizzl3_seven'].str.strip().str.lower().map(bool_map).astype('boolean')
df['invalid_text'] = df['text_clean'].eq('')
df['invalid_label'] = df['label'].isna()
df['invalid_flag'] = df['RDizzl3_seven'].isna()
df['missing_metadata'] = df[['source', 'prompt_name']].isna().any(axis=1)
df['replacement_character'] = df['text_clean'].str.contains('\ufffd', regex=False)
df['text_group'] = df['text_clean'].map(lambda x: hashlib.sha256(x.encode('utf-8')).hexdigest())
print('Văn bản thay đổi sau chuẩn hóa:', int(df['text'].ne(df['text_clean']).sum()))
display(df[['invalid_text', 'invalid_label', 'invalid_flag', 'missing_metadata',
            'replacement_character']].sum().rename('count').to_frame())


### 1.3. Trùng lặp, xung đột và bản ghi cần xem xét
Loại bản ghi trùng hoàn toàn theo các cột gốc đã chuẩn hóa (dùng `text_clean` thay `text`). Văn bản giống nhau nhưng khác metadata được giữ để không mất nguồn gốc và dùng chung `text_group` khi chia tập.

Văn bản rỗng, nhãn không hợp lệ, hoặc cùng văn bản nhưng có cả hai nhãn được đưa vào file cách ly. Thiếu metadata/boolean và ký tự thay thế chỉ được gắn cờ; không tự sửa hoặc xóa. Chưa thực hiện tìm bài gần trùng về ngữ nghĩa.


In [ ]:
label_counts = df.groupby('text_group')['label'].nunique()
conflict_groups = label_counts[label_counts > 1].index
df['label_conflict'] = df['text_group'].isin(conflict_groups)
dedup_cols = [c for c in raw.columns if c != 'text'] + ['text_clean']
df['duplicate_record'] = df.duplicated(subset=dedup_cols, keep='first')
quarantine_mask = df['invalid_text'] | df['invalid_label'] | df['label_conflict']
quarantine = df.loc[quarantine_mask].copy()
duplicates = df.loc[~quarantine_mask & df['duplicate_record']].copy()
clean = df.loc[~quarantine_mask & ~df['duplicate_record']].copy()
assert len(raw) == len(clean) + len(quarantine) + len(duplicates)
assert clean['label'].isin([0, 1]).all()
assert clean['text_clean'].ne('').all()
assert clean.groupby('text_group')['label'].nunique().le(1).all()
audit = pd.Series({
    'input_rows': len(raw), 'output_rows': len(clean),
    'quarantined_rows': len(quarantine), 'removed_duplicate_records': len(duplicates),
    'raw_duplicate_text_excess': int(raw['text'].duplicated().sum()),
    'normalized_duplicate_text_excess': int(df['text_clean'].duplicated().sum()),
    'conflicting_text_groups': len(conflict_groups),
    'changed_text_rows': int(df['text'].ne(df['text_clean']).sum())
}, name='count')
display(audit.to_frame())
display(quarantine[['row_id', 'label', 'source', 'invalid_text', 'invalid_label', 'label_conflict']].head(10))


### 1.4. Tạo đặc trưng mô tả và xuất dữ liệu
Đếm từ bằng tách khoảng trắng (không phải tokenizer của mô hình); đếm đoạn theo dòng trống. Ngưỡng độ dài 1%/99% chỉ dùng để chọn bài xem xét, không dùng để xóa dữ liệu.


In [ ]:
if clean.empty:
    raise ValueError('Không còn dữ liệu hợp lệ để EDA; kiểm tra bảng cách ly.')
text = clean['text_clean']
clean['char_count'] = text.str.len()
clean['word_count'] = text.str.split().str.len()
clean['paragraph_count'] = text.map(lambda s: len(re.split(r'\n\s*\n', s)))
clean['uppercase_ratio'] = text.map(lambda s: sum(c.isupper() for c in s) / max(sum(c.isalpha() for c in s), 1))
clean['punctuation_ratio'] = text.map(lambda s: sum(unicodedata.category(c).startswith('P') for c in s) / max(len(s), 1))
lo, hi = clean['word_count'].quantile([.01, .99])
clean['length_review'] = (clean['word_count'] < lo) | (clean['word_count'] > hi)
clean.to_csv(OUT / 'train_v2_drcat_02_clean.csv', index=False, encoding='utf-8-sig')
quarantine.to_csv(OUT / 'quarantine.csv', index=False, encoding='utf-8-sig')
duplicates.to_csv(OUT / 'removed_duplicates.csv', index=False, encoding='utf-8-sig')
audit.to_csv(OUT / 'cleaning_audit.csv', encoding='utf-8-sig')
print(f'Đã lưu kết quả tại: {OUT.resolve()}')
print(f'Ngưỡng xem xét độ dài: dưới {lo:.0f} hoặc trên {hi:.0f} từ')


## 2. Exploratory Data Analysis (EDA)
Các thống kê dưới đây dùng dữ liệu sau cleaning; mỗi dòng được tính là một quan sát, kể cả bài giống nhau nhưng khác metadata.
### 2.1. Phân bố nhãn, chủ đề và nguồn


In [ ]:
label_table = clean['label'].value_counts().sort_index().rename('count').to_frame()
label_table['percent'] = label_table['count'] / len(clean) * 100
display(label_table)
print('Số chủ đề:', clean['prompt_name'].nunique(), '| Số nguồn:', clean['source'].nunique())
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
label_table['count'].plot.bar(ax=axes[0], color=['#2563eb', '#f59e0b'], rot=0, title='Label distribution')
axes[0].set_ylabel('Rows')
clean['prompt_name'].fillna('(missing)').value_counts().sort_values().plot.barh(ax=axes[1], title='Rows by prompt')
fig.tight_layout()
fig.savefig(OUT / '01_label_prompt.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.2. Độ dài và đặc trưng phong cách theo nhãn
Histogram dùng chung khoảng chia để so sánh. Boxplot ẩn các điểm ngoại lệ cho dễ đọc; thống kê và dữ liệu vẫn giữ chúng.


In [ ]:
features = ['char_count', 'word_count', 'paragraph_count', 'uppercase_ratio', 'punctuation_ratio']
display(clean.groupby('label')[features].agg(['count', 'mean', 'median', 'std', 'min', 'max']))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
bins = np.linspace(clean.word_count.min(), max(clean.word_count.max(), clean.word_count.min() + 1), 51)
groups = list(clean.groupby('label'))
for label, group in groups:
    axes[0].hist(group.word_count, bins=bins, alpha=.5, density=True, label=f'Label {label}')
axes[0].set(title='Word count distribution', xlabel='Words', ylabel='Density')
axes[0].legend()
axes[1].boxplot([g.word_count for _, g in groups], showfliers=False)
axes[1].set_xticks(range(1, len(groups)+1), [str(k) for k, _ in groups])
axes[1].set(title='Word count by label', xlabel='Label', ylabel='Words')
fig.tight_layout()
fig.savefig(OUT / '02_text_length.png', dpi=150, bbox_inches='tight')
plt.show()
display(clean.loc[clean.length_review, ['row_id', 'label', 'source', 'word_count']].sort_values('word_count').head(10))


### 2.3. Quan hệ giữa nhãn, chủ đề, nguồn và nhóm boolean
Một nguồn chỉ có một nhãn có thể khiến mô hình học dấu hiệu của nguồn thay vì khái quát hóa. Metadata dùng để chẩn đoán; không mặc định đưa vào đầu vào mô hình.


In [ ]:
for col in ['source', 'prompt_name', 'RDizzl3_seven']:
    table = pd.crosstab(clean[col].astype('string').fillna('(missing)'), clean['label']).reindex(columns=[0, 1], fill_value=0)
    proportion = table.div(table.sum(axis=1), axis=0)
    display(table.assign(label_1_percent=proportion[1] * 100))
    table.to_csv(OUT / f'label_by_{col}.csv', encoding='utf-8-sig')
    ax = proportion.plot.barh(stacked=True, figsize=(10, max(3, len(table) * .35)), color=['#2563eb', '#f59e0b'])
    ax.set(title=f'Label proportions by {col}', xlabel='Proportion', ylabel=col, xlim=(0, 1))
    ax.legend(title='Label', bbox_to_anchor=(1.02, 1))
    plt.tight_layout()
    plt.savefig(OUT / f'03_label_by_{col}.png', dpi=150, bbox_inches='tight')
    plt.show()
display(clean.loc[clean.source.eq('train_essays') & clean.label.eq(1),
                  ['row_id', 'label', 'source', 'prompt_name', 'word_count']])


### 2.4. Từ và bigram phổ biến
Dùng biểu thức chính quy cho từ Latin, chuyển chữ thường chỉ trong phân tích này. Không loại stopwords để tránh áp đặt từ điển; vì vậy từ chức năng có thể đứng đầu. Tần suất chuẩn hóa trên 10.000 token/ngram giúp so sánh hai nhóm khác kích thước. Đây là thống kê mô tả, chưa chứng minh khả năng phân loại; cần kiểm tra ảnh hưởng của chủ đề.


In [ ]:
from collections import Counter
lexical_tables = []
for label, group in clean.groupby('label'):
    words, bigrams = Counter(), Counter()
    for value in group.text_clean:
        tokens = re.findall(r"[a-z]+(?:'[a-z]+)?", value.lower())
        words.update(tokens)
        bigrams.update(' '.join(pair) for pair in zip(tokens, tokens[1:]))
    for kind, counts in [('word', words), ('bigram', bigrams)]:
        total = sum(counts.values())
        top = pd.DataFrame(counts.most_common(20), columns=['term', 'count'])
        top['per_10000'] = top['count'] / max(total, 1) * 10000
        top['label'], top['kind'] = int(label), kind
        lexical_tables.append(top)
        print(f'Label {label} — {kind}')
        display(top.head(10))
pd.concat(lexical_tables, ignore_index=True).to_csv(OUT / 'top_words_bigrams.csv', index=False, encoding='utf-8-sig')


### 2.5. Tổng hợp và hướng sử dụng
Các số liệu dưới đây được tính lại khi chạy notebook. Chưa thực hiện kiểm định thống kê hay huấn luyện mô hình. Không coi các bài `train_essays` nhãn 1 là sai nếu chưa kiểm tra nguồn gốc.

Khi xây dựng mô hình: dùng `text_clean` làm đầu vào, giữ mọi bản ghi cùng `text_group` ở cùng một tập; kiểm tra bài gần trùng trước khi chia tập. So sánh đánh giá ngẫu nhiên có kiểm soát nhãn với đánh giá trên chủ đề/nguồn chưa thấy. Bộ vector hóa và mọi phép học từ dữ liệu phải chỉ được fit trên tập train. Các nguồn thuần một nhãn cần được phối hợp để tập đánh giá vẫn có cả hai lớp.


In [ ]:
source_table = pd.crosstab(clean.source.fillna('(missing)'), clean.label)
pure_sources = int(source_table.gt(0).sum(axis=1).eq(1).sum())
summary = {
    'rows_after_cleaning': len(clean),
    'label_counts': {str(k): int(v) for k, v in clean.label.value_counts().items()},
    'prompts': int(clean.prompt_name.nunique()),
    'sources': int(clean.source.nunique()),
    'sources_with_one_label': pure_sources,
    'median_words_by_label': {str(k): float(v) for k, v in clean.groupby('label').word_count.median().items()},
    'length_review_rows': int(clean.length_review.sum()),
    'duplicate_text_excess_retained': int(clean.text_group.duplicated().sum())
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
(OUT / 'eda_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('Hoàn tất. File gốc được giữ nguyên.')
